# Test the Metropolis package import

This developer example checks that Python can import `metropolis`, uses its runtime configuration helpers, and loads the Metropolis v1 address plan into a pandas table. The YAML file is read from this repository; it is not currently bundled as package data in the built wheel.

Before running, install the project and development dependencies with `uv sync --group dev`, then select that environment as this notebook's Python kernel.

In [ ]:
import metropolis
from metropolis import paths

print(f'Imported metropolis from: {metropolis.__file__}')
print(f'Repository root: {paths._BASE_DIR}')

In [ ]:
from metropolis.config import RuntimeConfig, resolve_device

runtime = RuntimeConfig()
print(runtime)
print(f'Available runtime device: {resolve_device(runtime.device)}')

## Load the testbed address plan

This treats the topology file as structured dataset metadata. Each row represents one LAN, router transit, WAN underlay, or reserved overlay network.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

address_plan_path = (
    paths._BASE_DIR
    / 'testbeds'
    / 'metropolis'
    / 'datasets'
    / 'water_treatment_v1'
    / 'topology'
    / 'address-plan.yaml'
)
with address_plan_path.open(encoding='utf-8') as address_plan_file:
    address_plan = yaml.safe_load(address_plan_file)

networks = pd.DataFrame(address_plan['networks'])
print(f"Testbed: {address_plan['testbed_id']}")
print(f"Networks: {len(networks)}")
display(networks[['id', 'site', 'zone', 'kind', 'cidr', 'gateway', 'vlan_id']])

In [ ]:
# A few useful summaries for exploring the topology.
display(networks.groupby(['site', 'zone'], dropna=False).size().rename('network_count'))
display(networks[networks['kind'].isin(['transit', 'underlay', 'overlay'])][['id', 'kind', 'cidr', 'purpose']])

In [ ]:
# Basic integrity checks that are useful while editing the topology data.
assert address_plan['testbed_id'] == 'metropolis'
assert networks['id'].is_unique
assert networks['cidr'].notna().all()
print('Import and address-plan checks passed.')